# Testing Combined H5 Processing

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import uproot
import sys
import seaborn as sns
from tqdm import tqdm
import networkx as nx
import matplotlib.cm as cm
import h5py
import logging

import atlasify as atl
from particle import Particle
atl.ATLAS = "ColliderML"

sys.path.append("/global/cfs/cdirs/m4958/usr/danieltm/ColliderML/software/OtherLibraries/pyedm4hep")
from pyedm4hep import EDM4hepEvent, EDM4hepEventBatch

## Single Event Testing

In [3]:
edm_input_file = "/pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/runs/0/edm4hep.root"
event = EDM4hepEvent(edm_input_file, event_index=0)

In [4]:
edm_input_file = "/pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_mini_pilot/ttbar/v7/runs/0/edm4hep.root"
event2 = EDM4hepEvent(edm_input_file, event_index=0)

In [9]:
print(f"""
Number of calo hits: {len(event.get_calo_hits_df())}
Number of particles: {len(event.get_particles_df())}
Number of tracker hits: {len(event.get_tracker_hits_df())}
""")


Number of calo hits: 1243953
Number of particles: 880327
Number of tracker hits: 240622



In [10]:
print(f"""
Number of calo hits: {len(event2.get_calo_hits_df())}
Number of particles: {len(event2.get_particles_df())}
Number of tracker hits: {len(event2.get_tracker_hits_df())}
""")


Number of calo hits: 1120404
Number of particles: 190319
Number of tracker hits: 217781



In [11]:
event2.get_calo_hits_df()

,cellID,energy,x,y,z,contribution_begin,contribution_end,detector,r,R,phi,theta,eta
0,152559394436048912,3.538352e-04,1190.580322,437.952515,2759.100098,0,9,ECalBarrelCollection,1268.575562,3036.760986,0.352486,0.430956,1.519249
1,4503814384687120,5.628845e-05,-274.604279,-1329.300415,81.599998,9,10,ECalBarrelCollection,1357.367676,1359.818237,-1.774509,1.510752,0.060080
2,1970496644814864,2.591849e-05,-333.317444,-1337.777222,35.700001,10,11,ECalBarrelCollection,1378.676270,1379.138428,-1.814982,1.544908,0.025892
3,5348204955310096,3.629815e-04,-325.826416,-1346.346069,96.900002,11,18,ECalBarrelCollection,1385.211426,1388.596558,-1.808239,1.500957,0.069896
4,64176406367602704,8.606262e-05,601.779175,-1106.322876,1162.800049,18,19,ECalBarrelCollection,1259.400024,1714.115601,-1.072613,0.825258,0.826083
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1120399,5629340626780692,2.156135e-09,1199.873169,342.497345,-3698.500000,5346121,5346123,HCalEndcapCollection,1247.798096,3903.319336,0.278050,2.816205,-1.807007
1120400,1407215976088084,5.042664e-04,1112.082520,-98.856026,-3647.500000,5346123,5346129,HCalEndcapCollection,1116.467651,3814.545410,-0.088660,2.844557,-1.899660
1120401,844300382437908,2.557875e-06,864.988586,-110.881462,-3698.500000,5346129,5346132,HCalEndcapCollection,872.066467,3799.921387,-0.127493,2.910033,-2.151582
1120402,5066390673457684,1.597710e-08,1188.167725,283.650238,-3851.500000,5346132,5346133,HCalEndcapCollection,1221.556396,4040.575684,0.234343,2.834465,-1.865733


In [12]:
event2.get_calo_contributions_df()

,PDG,energy,time,step_x,step_y,step_z,particle_id,cellID,x,y,z,detector
0,0,1.297844e-05,2562.023438,0.0,0.0,0.0,75414,0,1190.580322,437.952515,2759.100098,ECalBarrelCollection
1,0,8.172441e-06,2562.023438,0.0,0.0,0.0,75414,0,1190.580322,437.952515,2759.100098,ECalBarrelCollection
2,0,4.534445e-06,2562.023438,0.0,0.0,0.0,75414,0,1190.580322,437.952515,2759.100098,ECalBarrelCollection
3,0,1.256829e-05,2562.023682,0.0,0.0,0.0,75414,0,1190.580322,437.952515,2759.100098,ECalBarrelCollection
4,0,1.282355e-05,2562.023926,0.0,0.0,0.0,75414,0,1190.580322,437.952515,2759.100098,ECalBarrelCollection
...,...,...,...,...,...,...,...,...,...,...,...,...
5346129,0,9.253779e-07,187.780411,0.0,0.0,0.0,1036,201624,864.988586,-110.881462,-3698.500000,HCalEndcapCollection
5346130,0,1.031067e-06,188.810455,0.0,0.0,0.0,1036,201624,864.988586,-110.881462,-3698.500000,HCalEndcapCollection
5346131,0,6.014303e-07,189.794922,0.0,0.0,0.0,1036,201624,864.988586,-110.881462,-3698.500000,HCalEndcapCollection
5346132,0,1.597710e-08,1742.137573,0.0,0.0,0.0,1036,201625,1188.167725,283.650238,-3851.500000,HCalEndcapCollection


## Combined multi-output conversion (single open per run)

This workflow opens each `edm4hep.root` once per run and produces multiple H5 outputs (truth/particles and reco/tracker_hits) in one pass, reusing preloaded DataFrames to avoid repeated IO.


In [2]:
from pathlib import Path
import logging

# Reuse postprocessing helpers
sys.path.append("/global/cfs/cdirs/m4958/usr/danieltm/ColliderML/software/colliderml_dev/scripts/postprocessing")
from convert_particles import build_particles_df_with_parents_and_vertex, write_particles_with_selection
from convert_digihits import process_event_for_digihits, write_digihits_with_selection
from utils.path_utils import get_run_paths, make_dir
from utils.track_utils import load_root_file

from convert_all import convert_all


In [3]:
# Configurable column selection for H5 outputs
particles_columns_keep = [
    "particle_id", "pdg_id", "mass", "energy", "charge",
    "vx", "vy", "vz", "time", "px", "py", "pz",
    "num_tracker_hits", "num_calo_hits", "vertex_primary", "parent_id",
]
digihits_columns_keep = [
    "x", "y", "z", "time", "particle_id",
    "cell_id", "detector", "event_id",
]


In [4]:
# Config for this combined test (aligns with your production YAMLs)
campaign = "full_pileup_pilot"
dataset = "ttbar"
version = "v2"

base_root = Path("/pscratch/sd/d/danieltm/ColliderML/simulation")
output_base_dir = Path("./h5_testing/v2")  # unified root like scripts

logging.basicConfig(
    level=logging.DEBUG,  # show DEBUG and above
    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,  # override prior configs in this kernel
)

config = {
    "chunk_size": 100,
    "run_size": 128,
    "max_chunks": 1,
    "campaign": "full_pileup_pilot",
    "dataset": "ttbar",
    "version": "v2",
    "common": {
        "output_base_dir": base_root,
    },
    "objects": ["particles", "tracker_hits"],
    "particles_columns_keep": particles_columns_keep,
    "digihits_columns_keep": digihits_columns_keep,
    "min_tracker_hits": 1,
    "h5_output_dir": output_base_dir,
}


In [ ]:
convert_all(config)

2025-09-11 04:14:06,677 - DEBUG - convert_all - Starting conversion with config: campaign=full_pileup_pilot, dataset=ttbar, version=v2
2025-09-11 04:14:06,678 - DEBUG - convert_all - Input base directory: /pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2
2025-09-11 04:14:06,679 - DEBUG - convert_all - Output base directory: h5_testing/v2
2025-09-11 04:14:06,679 - DEBUG - convert_all - Chunk size: 100, Run size: 128
2025-09-11 04:14:06,679 - DEBUG - convert_all - Objects to convert: ['particles', 'tracker_hits']
2025-09-11 04:14:06,679 - DEBUG - convert_all - Dataset base path: full_pileup_pilot/ttbar/v2
2025-09-11 04:14:06,680 - DEBUG - convert_all - Importing required modules completed
2025-09-11 04:14:06,689 - DEBUG - convert_all - Retrieved 109 run directories from /pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2
2025-09-11 04:14:06,694 - DEBUG - convert_all - Created output directories: particles=h5_testing/v2/full_pileup_pilot/ttbar/v

2025-09-11 04:14:06,779 - DEBUG - fsspec.local - open file: /pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/runs/0/particles.root
2025-09-11 04:14:06,780 - DEBUG - fsspec.local - open file: /pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/runs/0/particles.root
2025-09-11 04:14:06,790 - DEBUG - fsspec.local - open file: /pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/runs/0/particles.root
2025-09-11 04:14:06,799 - DEBUG - fsspec.local - open file: /pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/runs/0/particles.root
2025-09-11 04:14:06,801 - DEBUG - fsspec.local - open file: /pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/runs/0/particles.root
2025-09-11 04:14:06,808 - DEBUG - fsspec.local - open file: /pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/runs/0/particles.root
2025-09-11 04:14:06,850 - DEBUG - asyncio - Using selector: EpollSelec

## Test

In [ ]:
def load_all_particles(h5_path):
    with h5py.File(h5_path, 'r') as f:
        frames = []
        for ev_name in f['events'].keys():
            ev_id = int(ev_name.split('_')[1])
            arr = f['events'][ev_name]['particles'][:]
            df = pd.DataFrame(arr)
            df['event_id'] = ev_id
            frames.append(df)
        return pd.concat(frames, ignore_index=True)

def load_all_digihits(h5_path):
    with h5py.File(h5_path, 'r') as f:
        frames = []
        for ev_name in f['events'].keys():
            ev_id = int(ev_name.split('_')[1])
            arr = f['events'][ev_name]['measurements'][:]
            df = pd.DataFrame(arr)
            df['event_id'] = ev_id
            frames.append(df)
        return pd.concat(frames, ignore_index=True)

In [22]:
particles_df = load_all_particles(particles_file)
digihits_df = load_all_digihits(digihits_file)

In [23]:
particles_df

,subentry,PDG,generatorStatus,simulatorStatus,charge,time,mass,vx,vy,vz,...,endpoint_z,parents_begin,parents_end,daughters_begin,daughters_end,vr,endpoint_r,energy,kinetic_energy,event_id
0,0,2212,4,0,1.000000,0.000000,0.938270,0.000000,0.000000,0.000000,...,-89.848333,0,0,0,4,0.000000,0.010252,7000.000000,6999.061730,0
1,1,-2,61,0,-0.666667,-0.765510,0.000000,0.009780,-0.003074,-89.848333,...,-89.848333,0,1,4,5,0.010252,0.010252,1028.604778,1028.604778,0
2,2,2,63,0,0.666667,-0.765510,0.330000,0.009780,-0.003074,-89.848333,...,-89.848333,1,2,5,6,0.010252,0.010252,2208.906631,2208.576631,0
3,3,2103,63,0,0.333333,-0.765510,0.771330,0.009780,-0.003074,-89.848333,...,-89.848333,2,3,6,7,0.010252,0.010252,1147.773510,1147.002180,0
4,4,2,63,0,0.666667,-0.765510,0.330000,0.009780,-0.003074,-89.848333,...,-89.848333,3,4,7,8,0.010252,0.010252,2614.671253,2614.341253,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
880322,880322,-11,0,1426063360,1.000000,9.437787,0.000511,-72.673144,1323.954928,2621.724743,...,2626.872701,1042284,1042285,1042289,1042289,1325.947977,1324.233574,0.007907,0.007396,0
880323,880323,-11,0,1426063360,1.000000,9.349813,0.000511,-64.631563,1297.143378,2596.949831,...,2594.640344,1042285,1042286,1042289,1042289,1298.752548,1293.777956,0.029932,0.029421,0
880324,880324,11,0,1426063360,-1.000000,8859.554688,0.000511,1249.222492,-386.984011,2908.253596,...,2908.509053,1042286,1042287,1042289,1042289,1307.789532,1309.365171,0.002381,0.001870,0
880325,880325,2112,0,1426063360,0.000000,25.586529,0.939565,1289.538138,400.301914,2893.576084,...,2275.590648,1042287,1042288,1042289,1042289,1350.240805,1255.805389,0.940569,0.001004,0


In [ ]:
particles_df.columns

Index(['subentry', 'PDG', 'generatorStatus', 'simulatorStatus', 'charge',
       'time', 'mass', 'vx', 'vy', 'vz', 'px', 'py', 'pz', 'endpoint_x',
       'endpoint_y', 'endpoint_z', 'parents_begin', 'parents_end',
       'daughters_begin', 'daughters_end', 'vr', 'endpoint_r', 'energy',
       'kinetic_energy', 'event_id'],
      dtype='object')

In [ ]:
digihits_df

,x,y,z,volume_id,layer_id,surface_id,true_x,true_y,true_z,time,px,py,pz,particle_id,cell_id,detector,e_dep,path_length,event_id
0,85.607491,3.613425,-1515.599976,16,4,1,85.631775,3.625161,-1515.599976,35.864868,-0.127980,0.013338,-0.021521,703493,67494740071,NaN,0.000370,0.821243,0
1,92.881134,-3.039804,-1515.599976,16,4,1,92.851669,-3.027657,-1515.599976,13.531095,0.165640,-0.054582,-2.974995,67930,1023410279,NaN,0.000036,0.125214,0
2,66.319473,-7.858758,-1515.599976,16,4,1,66.317802,-7.857886,-1515.599976,9.396918,0.268940,0.010043,-6.876966,74845,2634023015,NaN,0.000029,0.125096,0
3,65.872543,-3.477791,-1515.599976,16,4,1,65.890350,-3.451361,-1515.599976,8655.912109,-0.011574,-0.037811,-0.014304,669642,1157628007,NaN,0.000116,0.369049,0
4,92.184578,-4.563632,-1515.599976,16,4,1,92.188545,-4.542456,-1515.599976,6.495801,-0.001369,0.025355,-0.071781,375396,1526726759,NaN,0.000101,0.132867,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
240609,406.731476,903.742432,3009.500000,30,12,192,421.881561,942.290344,3009.500000,7.946029,0.693225,0.425183,2.162005,77697,858994544990,NaN,0.000081,0.267095,0
240610,325.683228,935.429993,3009.500000,30,12,192,306.645935,886.782227,3009.500000,11.627073,0.005575,0.012120,-0.007825,343955,15955804590430,NaN,0.000144,0.522783,0
240611,353.611938,924.510681,3009.500000,30,12,192,331.738159,868.459473,3009.500000,10.063175,0.001385,0.003332,-0.001101,573373,16819093016926,NaN,0.000224,0.663010,0
240612,435.766296,892.390686,3009.500000,30,12,192,446.899231,921.189087,3009.500000,7.529224,-0.001016,-0.001745,0.001744,759736,1748052775262,NaN,0.000116,0.420584,0
